In [1]:
import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

CSV_PATH = Path(r"D:\Tankdaten\rq_brand_vs_free\brand_vs_free_yearly_stats.csv")

In [2]:
df = pl.read_csv(CSV_PATH).filter(pl.col("year") < 2026)

brand = df.filter(pl.col("station_type") == "brand_station").sort("year")
free  = df.filter(pl.col("station_type") == "free_station").sort("year")

print(df)

shape: (24, 6)
┌───────────────┬──────┬────────────────────┬─────────────┬──────────┬──────────┐
│ station_type  ┆ year ┆ price_update_count ┆ diesel_mean ┆ e5_mean  ┆ e10_mean │
│ ---           ┆ ---  ┆ ---                ┆ ---         ┆ ---      ┆ ---      │
│ str           ┆ i64  ┆ i64                ┆ f64         ┆ f64      ┆ f64      │
╞═══════════════╪══════╪════════════════════╪═════════════╪══════════╪══════════╡
│ brand_station ┆ 2014 ┆ 13331083           ┆ 1.332714    ┆ 1.521487 ┆ 1.481391 │
│ brand_station ┆ 2015 ┆ 29683865           ┆ 1.166947    ┆ 1.389436 ┆ 1.369454 │
│ brand_station ┆ 2016 ┆ 37522628           ┆ 1.084146    ┆ 1.304853 ┆ 1.284594 │
│ brand_station ┆ 2017 ┆ 49980468           ┆ 1.161457    ┆ 1.368261 ┆ 1.345339 │
│ brand_station ┆ 2018 ┆ 55814952           ┆ 1.286967    ┆ 1.453957 ┆ 1.430943 │
│ …             ┆ …    ┆ …                  ┆ …           ┆ …        ┆ …        │
│ free_station  ┆ 2021 ┆ 16034776           ┆ 1.376616    ┆ 1.570909 ┆ 1.514974 │
│

## Mean Fuel Prices: Brand vs. Free Stations (per year)

In [3]:
fuels = [("diesel_mean", "Diesel"), ("e5_mean", "E5"), ("e10_mean", "E10")]

fig = make_subplots(rows=1, cols=3, shared_yaxes=True,
                    subplot_titles=[label for _, label in fuels])

for col_i, (col, label) in enumerate(fuels, start=1):
    show_legend = col_i == 1
    fig.add_trace(go.Scatter(
        x=brand["year"], y=brand[col].round(4),
        mode="lines+markers", name="Brand station",
        line=dict(color="steelblue", width=2),
        legendgroup="brand", showlegend=show_legend,
        hovertemplate="%{x}: %{y:.3f} €/L",
    ), row=1, col=col_i)
    fig.add_trace(go.Scatter(
        x=free["year"], y=free[col].round(4),
        mode="lines+markers", name="Free station",
        line=dict(color="darkorange", width=2, dash="dash"),
        legendgroup="free", showlegend=show_legend,
        hovertemplate="%{x}: %{y:.3f} €/L",
    ), row=1, col=col_i)

fig.update_yaxes(ticksuffix=" €", col=1)
fig.update_layout(
    title="Mean Fuel Prices: Brand vs. Free Stations (Germany)",
    height=450, width=1000,
    hovermode="x unified",
)
fig.show()

## Price Difference: Brand minus Free (per year)

In [4]:
years = brand["year"].to_list()

colors = {"Diesel": "steelblue", "E5": "darkorange", "E10": "seagreen"}

fig = go.Figure()
fig.add_hline(y=0, line_color="black", line_width=1)

for col, label in fuels:
    diff_ct = [(b - f) * 100 for b, f in zip(brand[col].to_list(), free[col].to_list())]
    fig.add_trace(go.Scatter(
        x=years, y=[round(v, 3) for v in diff_ct],
        mode="lines+markers", name=label,
        line=dict(color=colors[label], width=2),
        hovertemplate="%{x}: %{y:+.2f} ct/L",
    ))
    avg = sum(diff_ct) / len(diff_ct)
    print(f"{label}: avg difference = {avg:+.2f} ct/L (brand vs. free)")

fig.update_layout(
    title="Price Difference: Brand − Free (positive = brand is more expensive)",
    xaxis_title="Year",
    yaxis_title="Difference (ct/L)",
    yaxis_ticksuffix=" ct",
    hovermode="x unified",
    height=450, width=900,
)
fig.show()

Diesel: avg difference = +1.52 ct/L (brand vs. free)
E5: avg difference = +1.40 ct/L (brand vs. free)
E10: avg difference = +1.28 ct/L (brand vs. free)


## Data volume: price updates per year

In [5]:
fig = go.Figure([
    go.Bar(name="Brand", x=brand["year"], y=(brand["price_update_count"] / 1e6).round(2),
           marker_color="steelblue", hovertemplate="%{x}: %{y:.1f}M updates"),
    go.Bar(name="Free",  x=free["year"],  y=(free["price_update_count"]  / 1e6).round(2),
           marker_color="darkorange", hovertemplate="%{x}: %{y:.1f}M updates"),
])
fig.update_layout(
    barmode="group",
    title="Price Updates per Year (data volume)",
    xaxis_title="Year",
    yaxis_title="Updates (Million)",
    yaxis_ticksuffix="M",
    hovermode="x unified",
    height=400, width=900,
)
fig.show()